In [ ]:
from pathlib import Path
import sys

import torch
from PIL import Image
from torchvision import transforms

import matplotlib.pyplot as plt

ROOT_DIR = Path.cwd().parent
sys.path.insert(0, str(ROOT_DIR / "src"))

from model import YOLOv1
from inference import predict, decode_predictions, non_max_suppression

CHECKPOINT_PATH = ROOT_DIR / "checkpoints" / "yolov1_best.pth"

IMAGE_SIZE = 448

print(f"Project root: {ROOT_DIR}")
print(f"Checkpoint exists: {CHECKPOINT_PATH.exists()}")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])

print(f"Using device: {device}")

In [ ]:
if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"Checkpoint not found: {CHECKPOINT_PATH}")

net = YOLOv1().to(device)

state_dict = torch.load(
    CHECKPOINT_PATH,
    map_location=device,
    weights_only=True,
)

net.load_state_dict(state_dict)
net.eval()

print("YOLOv1 model loaded successfully.")

In [ ]:
# Select the VOC2012 dataset directory and its image folder
VOC_DIR = ROOT_DIR / "data" / "VOCdevkit" / "VOC2012"
IMAGE_DIR = VOC_DIR / "JPEGImages"

# Retrieve all image paths in alphabetical order
image_paths = sorted(IMAGE_DIR.glob("*.jpg"))

# Stop execution if no images are found
if not image_paths:
    raise FileNotFoundError(f"No images found in {IMAGE_DIR}")

# Select up to four images for inference
NUM_IMAGES = min(4, len(image_paths))
selected_paths = image_paths[:NUM_IMAGES]

# Load the selected images and convert them into model input tensors
images = []
image_tensors = []

for image_path in selected_paths:
    image = Image.open(image_path).convert("RGB")
    images.append(image)
    image_tensors.append(transform(image))

# Stack the image tensors into a batch and move it to the selected device
batch = torch.stack(image_tensors).to(device)

# Display the number of images loaded and the batch dimensions
print(f"Loaded {len(images)} images")
print(f"Input batch shape: {batch.shape}")

# Create a figure to visualize the selected images
fig, axes = plt.subplots(1, len(images), figsize=(5 * len(images), 5))

# Ensure axes is iterable when displaying a single image
if len(images) == 1:
    axes = [axes]

# Display each image with its filename
for ax, image, path in zip(axes, images, selected_paths):
    ax.imshow(image.resize((IMAGE_SIZE, IMAGE_SIZE)))
    ax.set_title(path.name)
    ax.axis("off")

# Adjust the layout and display the figure
plt.tight_layout()
plt.show()

In [ ]:
# Define the confidence threshold for filtering detections and the IoU threshold for NMS
CONF_THRESHOLD = 0.25
IOU_THRESHOLD = 0.5

# Run inference on the batch of images and apply confidence filtering and NMS
detections = predict(
    net,
    batch,
    conf_threshold=CONF_THRESHOLD,
    iou_threshold=IOU_THRESHOLD,
)

# Display the number of detections and their details for each image
for path, image_detections in zip(selected_paths, detections):
    print(f"\n{path.name}: {len(image_detections)} detections")

    # Print the predicted class, confidence score, and bounding box for each detection
    for detection in image_detections:
        print(
            f"Class: {detection['class_id']} | "
            f"Score: {detection['score']:.3f} | "
            f"Box: {detection['box']}"
        )

In [ ]:
# Define the class names corresponding to the 20 PASCAL VOC classes
VOC_CLASSES = [
    "aeroplane", "bicycle", "bird", "boat", "bottle",
    "bus", "car", "cat", "chair", "cow", "diningtable",
    "dog", "horse", "motorbike", "person", "pottedplant",
    "sheep", "sofa", "train", "tvmonitor"
]

# Create a figure to display the images and their predicted detections
fig, axes = plt.subplots(1, len(images), figsize=(6 * len(images), 6))

# Ensure axes is iterable when displaying a single image
if len(images) == 1:
    axes = [axes]

# Display each image and its detections
for ax, image, image_detections in zip(axes, images, detections):
    ax.imshow(image.resize((IMAGE_SIZE, IMAGE_SIZE)))

    # Draw a bounding box and label for each detection
    for detection in image_detections:
        class_id = detection["class_id"]
        score = detection["score"]
        x_center, y_center, width, height = detection["box"]

        # Convert box center coordinates to top-left coordinates
        x_min = x_center - width / 2
        y_min = y_center - height / 2

        # Create and display the bounding box
        rect = plt.Rectangle(
            (x_min, y_min),
            width,
            height,
            fill=False,
            edgecolor="lime",
            linewidth=2,
        )
        ax.add_patch(rect)

        # Display the predicted class name and confidence score
        ax.text(
            x_min,
            max(0, y_min - 5),
            f"{VOC_CLASSES[class_id]} {score:.2f}",
            color="white",
            fontsize=9,
            bbox=dict(facecolor="green", alpha=0.7, pad=2),
        )

    # Display the number of detections and hide the axes
    ax.set_title(f"Detections: {len(image_detections)}")
    ax.axis("off")

# Adjust spacing and display the final visualization
plt.tight_layout()
plt.show()

In [ ]:
import xml.etree.ElementTree as ET
import matplotlib.patches as patches

# Display ground-truth boxes and predicted boxes
fig, axes = plt.subplots(1, len(images), figsize=(6 * len(images), 6))

if len(images) == 1:
    axes = [axes]

for ax, image, path, image_detections in zip(
    axes, images, selected_paths, detections
):
    ax.imshow(image)
    width, height = image.size

    # Ground-truth bounding boxes (blue)
    annotation_path = VOC_DIR / "Annotations" / f"{path.stem}.xml"
    root = ET.parse(annotation_path).getroot()

    for obj in root.findall("object"):
        class_name = obj.findtext("name")
        bbox = obj.find("bndbox")

        xmin = float(bbox.findtext("xmin"))
        ymin = float(bbox.findtext("ymin"))
        xmax = float(bbox.findtext("xmax"))
        ymax = float(bbox.findtext("ymax"))

        rect = patches.Rectangle(
            (xmin, ymin),
            xmax - xmin,
            ymax - ymin,
            linewidth=2,
            edgecolor="blue",
            facecolor="none",
        )
        ax.add_patch(rect)

        ax.text(
            xmin,
            max(0, ymin - 5),
            f"GT: {class_name}",
            color="white",
            fontsize=8,
            bbox=dict(facecolor="blue", alpha=0.7, pad=2),
        )

    # Predicted bounding boxes (green)
    scale_x = width / IMAGE_SIZE
    scale_y = height / IMAGE_SIZE

    for detection in image_detections:
        class_id = detection["class_id"]
        score = detection["score"]
        x_center, y_center, box_width, box_height = detection["box"]

        # Convert coordinates from 448x448 to original image size
        x_center *= scale_x
        y_center *= scale_y
        box_width *= scale_x
        box_height *= scale_y

        xmin = x_center - box_width / 2
        ymin = y_center - box_height / 2

        rect = patches.Rectangle(
            (xmin, ymin),
            box_width,
            box_height,
            linewidth=2,
            edgecolor="lime",
            facecolor="none",
        )
        ax.add_patch(rect)

        ax.text(
            xmin,
            max(0, ymin + 12),
            f"Pred: {VOC_CLASSES[class_id]} {score:.2f}",
            color="black",
            fontsize=8,
            bbox=dict(facecolor="lime", alpha=0.7, pad=2),
        )

    ax.set_title(path.name)
    ax.axis("off")

# Legend
legend_handles = [
    patches.Patch(
        edgecolor="blue",
        facecolor="none",
        label="Ground truth",
    ),
    patches.Patch(
        edgecolor="lime",
        facecolor="none",
        label="Prediction",
    ),
]

fig.legend(handles=legend_handles, loc="lower center", ncol=2)
plt.tight_layout(rect=[0, 0.08, 1, 1])
plt.show()